In [1]:
import pandas as pd 
import numpy as np 
import torch 
import json
import os
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from datetime import datetime
import sys
sys.path.append('../')
from utils.model import ToxicityModel

/mnt/windows/wheelchair-dev/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = 'distilbert-base-uncased'
THRESHOLD = 0.5
Y_MIN, Y_MAX = 0.0, 1.0
SIZE = 32
MAX_LEN = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def score_to_category(score):
    if score <= 0.20:   return "Safe"
    elif score <= 0.45: return "Ambiguous"
    elif score <= 0.75: return "Implicit/Covert Hate"
    else:               return "Explicit Toxicity"

In [4]:
model = ToxicityModel(MODEL_NAME).to(DEVICE)
model.load_state_dict(torch.load('../models/distilbert_toxicity_best.pt',
                                  map_location=DEVICE, weights_only=True))
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def predict(text):
    enc = tokenizer(str(text), return_tensors='pt', max_length=MAX_LEN,
                    truncation=True, padding='max_length')
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            score, _, _ = model(enc['input_ids'].to(DEVICE),
                                enc['attention_mask'].to(DEVICE))
    score = score.item()
    return {
        'text':       text,
        'raw_score':  score,
        'category':   score_to_category(score),
        'flag':       score >= THRESHOLD
    }

print(f"Model loaded on {DEVICE}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8315.93it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on cuda


In [5]:
# MFT Tests
mft_tests = [
    # Explicit hate — expect >0.75
    {"text": "eat shit and die you worthless piece of trash",       "expect_min": 0.75, "expect_max": 1.00, "type": "MFT", "capability": "Explicit Hate"},
    {"text": "i hope you get cancer and die tomorrow",              "expect_min": 0.75, "expect_max": 1.00, "type": "MFT", "capability": "Explicit Hate"},
    {"text": "kill yourself nobody wants you here",                 "expect_min": 0.75, "expect_max": 1.00, "type": "MFT", "capability": "Explicit Hate"},
    {"text": "you are a worthless nigger and should be exterminated","expect_min": 0.75, "expect_max": 1.00, "type": "MFT", "capability": "Explicit Hate"},
    {"text": "go drink bleach you stupid whore",                    "expect_min": 0.75, "expect_max": 1.00, "type": "MFT", "capability": "Explicit Hate"},

    # Safe / Benign — expect <0.20
    {"text": "I had a really wonderful day at the park today",      "expect_min": 0.00, "expect_max": 0.20, "type": "MFT", "capability": "Safe Benign"},
    {"text": "Thank you so much for your help, I really appreciate it", "expect_min": 0.00, "expect_max": 0.20, "type": "MFT", "capability": "Safe Benign"},
    {"text": "The weather is beautiful today, I love sunny days",   "expect_min": 0.00, "expect_max": 0.20, "type": "MFT", "capability": "Safe Benign"},
    {"text": "Your Character is so strong and I admire you",        "expect_min": 0.00, "expect_max": 0.20, "type": "MFT", "capability": "Safe Benign"},
    {"text": "I love you in the whole wide world",                  "expect_min": 0.00, "expect_max": 0.20, "type": "MFT", "capability": "Safe Benign"},
]

mft_results = []
for test in mft_tests:
    r = predict(test['text'])
    passed = test['expect_min'] <= r['raw_score'] <= test['expect_max']
    mft_results.append({
        **test,
        'score':    r['raw_score'],
        'category': r['category'],
        'passed':   passed
    })

mft_df = pd.DataFrame(mft_results)
passed = mft_df['passed'].sum()
total  = len(mft_df)
print(f"MFT Results: {passed}/{total} passed ({passed/total*100:.1f}%)\n")
print(mft_df[['capability', 'text', 'score', 'category', 'passed']].to_string(index=False))

[transformers] `sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


MFT Results: 7/10 passed (70.0%)

   capability                                                    text    score             category  passed
Explicit Hate           eat shit and die you worthless piece of trash 0.954102    Explicit Toxicity    True
Explicit Hate                  i hope you get cancer and die tomorrow 0.684570 Implicit/Covert Hate   False
Explicit Hate                     kill yourself nobody wants you here 0.560059 Implicit/Covert Hate   False
Explicit Hate   you are a worthless nigger and should be exterminated 0.845215    Explicit Toxicity    True
Explicit Hate                        go drink bleach you stupid whore 0.568848 Implicit/Covert Hate   False
  Safe Benign          I had a really wonderful day at the park today 0.017990                 Safe    True
  Safe Benign Thank you so much for your help, I really appreciate it 0.016983                 Safe    True
  Safe Benign       The weather is beautiful today, I love sunny days 0.019836                 Safe   

In [6]:
#Invariance Tests
identity_groups = [
    ("Muslim", "Christian"), ("Muslim", "Jewish"), ("Muslim", "Hindu"),
    ("gay", "straight"),     ("gay", "lesbian"),   ("gay", "bisexual"),
    ("Black", "White"),      ("Black", "Asian"),   ("Black", "Latino"),
]

inv_templates = [
    "My new neighbor is a {} man.",
    "I work with a {} colleague every day.",
    "The {} community deserves respect.",
    "I'm a proud {}.",
    "A {} person moved into our street.",
]

inv_tests = []
for template in inv_templates:
    for group_a, group_b in identity_groups:
        text_a = template.format(group_a)
        text_b = template.format(group_b)
        inv_tests.append({"text": text_a, "group": group_a,
                          "pair": group_b, "template": template, "type": "INV"})
        inv_tests.append({"text": text_b, "group": group_b,
                          "pair": group_a, "template": template, "type": "INV"})

inv_results = []
for test in inv_tests:
    r = predict(test['text'])
    # INV: neutral identity mentions must stay Safe (<0.20)
    passed = r['raw_score'] < 0.20
    inv_results.append({
        **test,
        'score':    r['raw_score'],
        'category': r['category'],
        'passed':   passed
    })

inv_df = pd.DataFrame(inv_results)
passed = inv_df['passed'].sum()
total  = len(inv_df)
print(f"INV Results: {passed}/{total} passed ({passed/total*100:.1f}%)\n")

# Show failures only
failures = inv_df[~inv_df['passed']]
if len(failures) > 0:
    print("FAILURES:")
    print(failures[['group', 'text', 'score', 'category']].to_string(index=False))
else:
    print("All INV tests passed — no identity bias detected.")

# Score variance between group pairs (lower = more fair)
print("\nAverage score by identity group:")
print(inv_df.groupby('group')['score'].mean().sort_values(ascending=False).to_string())

INV Results: 90/90 passed (100.0%)

All INV tests passed — no identity bias detected.

Average score by identity group:
group
Black        0.096460
gay          0.072551
lesbian      0.066324
White        0.045811
Asian        0.034854
straight     0.034485
Muslim       0.033902
Jewish       0.033456
bisexual     0.030305
Hindu        0.029135
Latino       0.025903
Christian    0.020157


In [ ]:
#DIR Tests
dir_tests = [
    # Negation -> adding "not" should decrease score
    {
        "text_before": "I hate you and despise everything about you",
        "text_after":  "I do not hate you and I do not despise everything about you",
        "expect":      "decrease",
        "capability":  "Negation of Hate"
    },
    {
        "text_before": "You are a terrible and awful person",
        "text_after":  "You are not a terrible or awful person",
        "expect":      "decrease",
        "capability":  "Negation of Hate"
    },
    # Reclaimed language -> in-group solidarity should score lower
    {
        "text_before": "you stupid nigger get out",
        "text_after":  "we are taking back our power my nigga, stay strong",
        "expect":      "decrease",
        "capability":  "Reclaimed Language"
    },
    # Escalation -> adding threat should increase score
    {
        "text_before": "I really dislike you",
        "text_after":  "I really dislike you and I hope you die",
        "expect":      "increase",
        "capability":  "Threat Escalation"
    },
    {
        "text_before": "You are so annoying",
        "text_after":  "You are so annoying, go kill yourself",
        "expect":      "increase",
        "capability":  "Threat Escalation"
    },
    # Positive reframing -> should decrease score
    {
        "text_before": "you are such an idiot",
        "text_after":  "you made a mistake but you can do better",
        "expect":      "decrease",
        "capability":  "Positive Reframing"
    },
]

dir_results = []
for test in dir_tests:
    r_before = predict(test['text_before'])
    r_after  = predict(test['text_after'])
    delta    = r_after['raw_score'] - r_before['raw_score']

    if test['expect'] == "decrease":
        passed = delta < 0
    else:
        passed = delta > 0

    dir_results.append({
        'capability':    test['capability'],
        'expect':        test['expect'],
        'text_before':   test['text_before'],
        'score_before':  r_before['raw_score'],
        'text_after':    test['text_after'],
        'score_after':   r_after['raw_score'],
        'delta':         delta,
        'passed':        passed,
        'type':          'DIR'
    })

dir_df = pd.DataFrame(dir_results)
passed = dir_df['passed'].sum()
total  = len(dir_df)
print(f"DIR Results: {passed}/{total} passed ({passed/total*100:.1f}%)\n")
print(dir_df[['capability', 'score_before', 'score_after', 'delta', 'passed']].to_string(index=False))

DIR Results: 6/6 passed (100.0%)

        capability  score_before  score_after     delta  passed
  Negation of Hate      0.437988     0.257080 -0.180908    True
  Negation of Hate      0.322021     0.164795 -0.157227    True
Reclaimed Language      0.910645     0.416504 -0.494141    True
 Threat Escalation      0.109924     0.782227  0.672302    True
 Threat Escalation      0.312988     0.565918  0.252930    True
Positive Reframing      0.520996     0.104309 -0.416687    True


In [8]:
# Compile full report
summary = {
    "timestamp":    datetime.now().isoformat(),
    "model":        MODEL_NAME,
    "checkpoint":   "../models/distilbert_toxicity_best.pt",
    "results": {
        "MFT": {
            "passed":  int(mft_df['passed'].sum()),
            "total":   len(mft_df),
            "pass_rate": float(mft_df['passed'].mean())
        },
        "INV": {
            "passed":  int(inv_df['passed'].sum()),
            "total":   len(inv_df),
            "pass_rate": float(inv_df['passed'].mean()),
            "mean_score_by_group": inv_df.groupby('group')['score'].mean().to_dict()
        },
        "DIR": {
            "passed":  int(dir_df['passed'].sum()),
            "total":   len(dir_df),
            "pass_rate": float(dir_df['passed'].mean())
        }
    }
}

overall_passed = (mft_df['passed'].sum() + inv_df['passed'].sum() + dir_df['passed'].sum())
overall_total  = len(mft_df) + len(inv_df) + len(dir_df)
summary["overall"] = {
    "passed":    int(overall_passed),
    "total":     int(overall_total),
    "pass_rate": float(overall_passed / overall_total)
}

print("CHECKLIST BEHAVIORAL TESTING SUMMARY")
print(f"MFT  (Minimum Functionality): {summary['results']['MFT']['passed']}/{summary['results']['MFT']['total']} "
      f"({summary['results']['MFT']['pass_rate']*100:.1f}%)")
print(f"INV  (Invariance):            {summary['results']['INV']['passed']}/{summary['results']['INV']['total']} "
      f"({summary['results']['INV']['pass_rate']*100:.1f}%)")
print(f"DIR  (Directional):           {summary['results']['DIR']['passed']}/{summary['results']['DIR']['total']} "
      f"({summary['results']['DIR']['pass_rate']*100:.1f}%)")
print(f"\nOverall: {overall_passed}/{overall_total} ({overall_passed/overall_total*100:.1f}%)")


os.makedirs('../data/checklist', exist_ok=True)

mft_df.to_csv('../data/checklist/mft_results.csv', index=False)
inv_df.to_csv('../data/checklist/inv_results.csv', index=False)
dir_df.to_csv('../data/checklist/dir_results.csv', index=False)

with open('../data/checklist/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\nResults saved to ../data/checklist/")

CHECKLIST BEHAVIORAL TESTING SUMMARY
MFT  (Minimum Functionality): 7/10 (70.0%)
INV  (Invariance):            90/90 (100.0%)
DIR  (Directional):           6/6 (100.0%)

Overall: 103/106 (97.2%)

Results saved to ../data/checklist/


In [9]:
failed_mft = mft_df[~mft_df['passed']]
print(failed_mft[['capability', 'text', 'score', 'expect_min', 'expect_max']].to_string())

      capability                                    text     score  expect_min  expect_max
1  Explicit Hate  i hope you get cancer and die tomorrow  0.684570        0.75         1.0
2  Explicit Hate     kill yourself nobody wants you here  0.560059        0.75         1.0
4  Explicit Hate        go drink bleach you stupid whore  0.568848        0.75         1.0
